# Imports and Settings

In [9]:
import sys
'''
Добавляем в список путей поиска модулей папку src,
которая находится на уровень выше,
чтобы импортировать модули из этой папки.
'''
sys.path.append('../src')  
import logging
import pickle
from functools import partial

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from config import (
    CAT_FEATURES,
    CAST_TYPE_MAP,
    CLASSES_METRIC_LIST,
    DEMO_PIPELINE_PATH,
    DROP_LIST,
    DROP_LIST_ENC_PAYM_NORM_GROUP_SUMM_DIFF,
    DROP_LIST_MEAN_VALUE_FREQUENCY_FEATURE,
    FLOAT_DOWNCAST_COLUMNS_LIST,
    MEAN_FREQ_SOURCE_LIST,
    N_SPLITS,
    PARQUET_FILE_PATTERN,
    PARAMS_LIST,
    PRE_FEATURES,
    PROP_FEATURES_DICT,
    RAW_DATA_PATH,
    SAMPLE_FRAC,
    SAVE_FILE_EXTENSION,
    SEARCH_FILE_EXTENSION,
    SEED,
    SEED_SPLIT_DATASET,
    SHUFFLE,
    STRATIFY_COL,
    TARGET_PATH,
    TEST_PREDICT_PATH,
    THRESHOLD,
    TEMP_DATA_PATH,
    TRAIN_SIZE,
    WEIGHTS_LIST
)


from data_utils import (
    load_dataset,
    split_dataset_by_target,
    check_data_folder_and_count_files,
    make_file_path,
    save_predictions_with_id,
)

from preprocessing import (
    SampleMedianImputer,
    convert_all_to_numeric_preprocessing,
    cast_columns_by_map_preprocessing,
    drop_duplicates_preprocessing,
)

from feature_engineering import (
    rn_max_feature_pipeline,
    enc_paym_transcoding_pipeline,
    definite_value_proportion_features_pipeline,
    from_is_zero_prop_1_create_sum_prop_1_feature_pipeline,
    mean_value_frequency_feature_pipeline,
    enc_paym_norm_group_sum_diff_pipeline,
    pre_since_opened_sum_mean_repeated_pipeline,
    drop_columns_pipeline,
    drop_duplicates_pipeline
)

from classifier import CatBoostEnsembleClassifier

# Logging switch

In [2]:
"""
Переключатель логирования функций пайплайна.
При выборе опции'INFO' будут выводиться названия функций, 
названия исходных обрабатываемых  признаков 
и названия новых фичей.
В классификаторе будут выводиться названия методов,
этапы обучения ансамбля и гиперпараметры моделей ансамбля.
При выборе опции 'OFF' логи отключаются.
"""

log_level_input = input(
    """
Введите уровень логирования для pipeline функций
'INFO' для включения, 'OFF' для отключения
"""
).strip().upper()

if log_level_input == 'OFF':
    # Блокируем логи
    logging.disable(logging.CRITICAL)  
    print("Логирование pipeline функций отключено")

elif log_level_input == 'INFO':
    # Снимаем блокировку
    logging.disable(logging.NOTSET) 

    # Удаляем старые обработчики, чтобы basicConfig сработал
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s'
    )
    print("Логирование pipeline функций включено")

else:
    # Снимаем блокировку
    logging.disable(logging.NOTSET)  

    # Удаляем старые обработчики, чтобы basicConfig сработал
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s'
    )
    print("Неверный ввод, установлен режим INFO")

# Создаём объект логера
logger = logging.getLogger(__name__)


Введите уровень логирования для pipeline функций
'INFO' для включения, 'OFF' для отключения
 INFO


Логирование pipeline функций включено


# Load and split data

In [3]:
"""
Собираем исходный датасет из parquet файлов,  
скачиваем только необходимые колонки
"""

# Получаем количество файлов в папке с данными
files_count = check_data_folder_and_count_files(RAW_DATA_PATH, PARQUET_FILE_PATTERN)[1]

# Загружаем датасет
raw_data = load_dataset(
    path_to_dataset=RAW_DATA_PATH,
    num_parts_to_preprocess_at_once=1,
    num_parts_total=files_count,
    save_to_path=TEMP_DATA_PATH,
    verbose=True,
    columns=PRE_FEATURES,
    cast_type_map=CAST_TYPE_MAP
)

# Загружаем таргет
# Делим датасет и таргет на train/test
train_test_dict = split_dataset_by_target(
        dataset=raw_data,
        path_to_target=TARGET_PATH,
        train_size=TRAIN_SIZE,
        random_state=SEED_SPLIT_DATASET,
        stratify_col=STRATIFY_COL
    )

2025-09-15 18:13:41,858 - Starting check_data_folder_and_count_files : /Users/admin/DataSince/Machine Learning Junior/21_final_project/21_final_project_5/data/raw
2025-09-15 18:13:41,862 - Count of files in data folder: 12
2025-09-15 18:13:41,863 - Starting load_dataset function
Loading entire data:   0%|                               | 0/12 [00:00<?, ?it/s]2025-09-15 18:13:41,883 - Processing step 0
2025-09-15 18:13:41,884 - Starting load_data_chunks function
2025-09-15 18:13:41,885 - Found 12 dataset paths
2025-09-15 18:13:41,886 - Reading chunks:
2025-09-15 18:13:41,886 - /Users/admin/DataSince/Machine Learning Junior/21_final_project/21_final_project_5/data/raw/train_data_0.pq

Reading dataset with pandas: 100%|████████████████| 1/1 [00:03<00:00,  3.90s/it]
2025-09-15 18:13:49,036 - Finished load_data_chunks (read 1974724 rows)
2025-09-15 18:13:51,203 - Saved to "/Users/admin/DataSince/Machine Learning Junior/21_final_project/21_final_project_5/data/temp/processed_chunk_000.parquet

# Pipeline

In [4]:
# Создадим объект классификатора
classifier = CatBoostEnsembleClassifier(
    params_list=PARAMS_LIST,
    weights_list=WEIGHTS_LIST,
    threshold=THRESHOLD,
    cat_features=CAT_FEATURES,
    n_splits=N_SPLITS,
    seed=SEED,
    shuffle=SHUFFLE,
    logger=logger
)
classifier

CatBoostEnsembleClassifier(cat_features=[], logger=<Logger __main__ (INFO)>,
                           params_list=[{'auto_class_weights': 'Balanced',
                                         'bagging_temperature': 0.09710127579,
                                         'boosting_type': 'Plain',
                                         'border_count': 113, 'depth': 4,
                                         'eval_metric': 'AUC',
                                         'grow_policy': 'SymmetricTree',
                                         'iterations': 3000,
                                         'l2_leaf_reg': 8.005778243,
                                         'learning_rate': 0.03625476076,
                                         'loss_function': 'L...
                                         'l2_leaf_reg': 8.005778242558318,
                                         'learning_rate': 0.036254760756236626,
                                         'min_data_in_leaf': 5,
                                         'random_seed': 0,
                                         'random_strength': 8.209932298658357,
                                         'rsm': 0.7343256008238508,
                                         'subsample': 0.9882297325066979,
                                         'verbose': False}],
                           threshold=0.48276,
                           weights_list=[0.7558470790046106, 0.7541369034301536,
                                         0.7517121820944949, 0.7529815101963229,
                                         0.7508019090111355,
                                         0.7530959167473436])

In [5]:
# Соберём пайплайн обработки данных и обучения ансамбля моделей

# Создаём SampleMedianImputer для заполнения пустых значений медианами
imputer = SampleMedianImputer(sample_frac=SAMPLE_FRAC)

# Создаём паплайн препроцессинга данных
preprocessing_pipe = Pipeline([
    (
        'to_numeric',
        FunctionTransformer(
            convert_all_to_numeric_preprocessing
        )
    ),
    (
        'imputer', imputer
    ),
    (
        'cast_type',
        FunctionTransformer(
            partial(
                cast_columns_by_map_preprocessing,
                cast_type_map=CAST_TYPE_MAP
            )
        )
    ),
    (
        'drop_duplicates_preprocessing',
        FunctionTransformer(
            drop_duplicates_preprocessing
        )
    )
])

# Создаём основной пайплайн
main_pipe = Pipeline(
    [
        (
            'preprocessing',
            preprocessing_pipe
        ),
        (
            'create_rn_max_feature',
            FunctionTransformer(
                rn_max_feature_pipeline
            )
        ),
        (
            'enc_paym_transcoding',
            FunctionTransformer(
                enc_paym_transcoding_pipeline
            )
        ),
        (
            'from_enc_paym_create_normalized_group_sum_features_then_diff_features',
            FunctionTransformer(
                partial(
                    enc_paym_norm_group_sum_diff_pipeline,
                    drop_list=DROP_LIST_ENC_PAYM_NORM_GROUP_SUMM_DIFF
                )
            )
        ),
        (
            'create_mean_value_frequency_feature',
            FunctionTransformer(
                partial(
                    mean_value_frequency_feature_pipeline,
                    columns_list=MEAN_FREQ_SOURCE_LIST,
                    drop_list=DROP_LIST_MEAN_VALUE_FREQUENCY_FEATURE
                )
            )
        ),
        (
            'create_definite_value_proportion_features',
            FunctionTransformer(
                partial(
                    definite_value_proportion_features_pipeline,
                    features_dictionary=PROP_FEATURES_DICT,
                    float_downcast_columns_list=FLOAT_DOWNCAST_COLUMNS_LIST
                )
            )
        ),
        (
            'create_sum_prop_1_feature',
            FunctionTransformer(
                from_is_zero_prop_1_create_sum_prop_1_feature_pipeline
            )
        ),
        (
            'from_pre_since_opened_create_pre_since_opened_sum_mean_repeated',
            FunctionTransformer(
                pre_since_opened_sum_mean_repeated_pipeline
            )
        ),
        (
            'drop_temporary_source_columns',
            FunctionTransformer(
                partial(
                    drop_columns_pipeline,
                    columns_list=DROP_LIST
                )
            )
        ),
        (
            'drop_duplicates_and_id',
            FunctionTransformer(
                drop_duplicates_pipeline
            )
        ),
        (
            'classifier', classifier
        )

    ]
)

In [6]:
# Обучим пайплайн
main_pipe.fit(train_test_dict['X_train'], train_test_dict['y_train'])

2025-09-15 18:15:39,997 - FUNCTION convert_all_to_numeric_preprocessing
2025-09-15 18:15:50,033 - FUNCTION cast_columns_by_map_preprocessing
2025-09-15 18:15:51,174 - FUNCTION drop_duplicates_preprocessing
2025-09-15 18:16:52,616 - No duplicates found, no cleanup operation required.
2025-09-15 18:16:52,654 - FUNCTION rn_max_feature_pipeline
2025-09-15 18:17:01,646 - FUNCTION enc_paym_transcoding_pipeline
2025-09-15 18:17:05,840 - FUNCTION enc_paym_norm_group_sum_diff_pipeline
2025-09-15 18:17:05,841 - NEW temporary features
2025-09-15 18:17:06,900 - enc_paym_avg_1_all
2025-09-15 18:17:09,462 - enc_paym_avg_2_all
2025-09-15 18:17:12,140 - enc_paym_avg_0_this_year
2025-09-15 18:17:14,244 - enc_paym_avg_1_this_year
2025-09-15 18:17:16,265 - enc_paym_avg_0_last_year
2025-09-15 18:17:19,296 - NEW difference columns
enc_paym_avg_0_1_this_year_diff
enc_paym_avg_1_2_all_diff
enc_paym_avg_0_years_diff

2025-09-15 18:17:22,262 - DataFrame shape after drop(): (20931476, 35)
2025-09-15 18:17:22,53

Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('to_numeric',
                                  FunctionTransformer(func=<function convert_all_to_numeric_preprocessing at 0x14701fd80>)),
                                 ('imputer', SampleMedianImputer()),
                                 ('cast_type',
                                  FunctionTransformer(func=functools.partial(<function cast_columns_by_map_preprocessing at 0x147024400>, cast_type_map={'id': 'int32', 'rn': 'int8', 'pre_since_op...
                                                          'l2_leaf_reg': 8.005778242558318,
                                                          'learning_rate': 0.036254760756236626,
                                                          'min_data_in_leaf': 5,
                                                          'random_seed': 0,
                                                          'random_strength': 8.209932298658357,
                                                          'rsm': 0.7343256008238508,
                                                          'subsample': 0.9882297325066979,
                                                          'verbose': False}],
                                            threshold=0.48276,
                                            weights_list=[0.7558470790046106,
                                                          0.7541369034301536,
                                                          0.7517121820944949,
                                                          0.7529815101963229,
                                                          0.7508019090111355,
                                                          0.7530959167473436]))])

In [10]:
# Сохраним обученный пайплайн в файл
with open(DEMO_PIPELINE_PATH, 'wb') as file:
    pickle.dump(main_pipe, file)

In [11]:
# Загрузим обученный пайплайн
with open(DEMO_PIPELINE_PATH, 'rb') as file:
   main_pipe = pickle.load(file)
main_pipe

Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('to_numeric',
                                  FunctionTransformer(func=<function convert_all_to_numeric_preprocessing at 0x14701fd80>)),
                                 ('imputer', SampleMedianImputer()),
                                 ('cast_type',
                                  FunctionTransformer(func=functools.partial(<function cast_columns_by_map_preprocessing at 0x147024400>, cast_type_map={'id': 'int32', 'rn': 'int8', 'pre_since_op...
                                                          'l2_leaf_reg': 8.005778242558318,
                                                          'learning_rate': 0.036254760756236626,
                                                          'min_data_in_leaf': 5,
                                                          'random_seed': 0,
                                                          'random_strength': 8.209932298658357,
                                                          'rsm': 0.7343256008238508,
                                                          'subsample': 0.9882297325066979,
                                                          'verbose': False}],
                                            threshold=0.48276,
                                            weights_list=[0.7558470790046106,
                                                          0.7541369034301536,
                                                          0.7517121820944949,
                                                          0.7529815101963229,
                                                          0.7508019090111355,
                                                          0.7530959167473436]))])

In [12]:
# Предскажем вероятности классов
pred_proba = main_pipe.predict_proba(train_test_dict['X_test'])
pred_proba

2025-09-15 19:32:16,211 - FUNCTION convert_all_to_numeric_preprocessing
2025-09-15 19:32:17,851 - FUNCTION cast_columns_by_map_preprocessing
2025-09-15 19:32:18,117 - FUNCTION drop_duplicates_preprocessing
2025-09-15 19:32:24,019 - No duplicates found, no cleanup operation required.
2025-09-15 19:32:24,020 - FUNCTION rn_max_feature_pipeline
2025-09-15 19:32:25,237 - FUNCTION enc_paym_transcoding_pipeline
2025-09-15 19:32:26,224 - FUNCTION enc_paym_norm_group_sum_diff_pipeline
2025-09-15 19:32:26,225 - NEW temporary features
2025-09-15 19:32:26,494 - enc_paym_avg_1_all
2025-09-15 19:32:27,040 - enc_paym_avg_2_all
2025-09-15 19:32:27,611 - enc_paym_avg_0_this_year
2025-09-15 19:32:27,993 - enc_paym_avg_1_this_year
2025-09-15 19:32:28,362 - enc_paym_avg_0_last_year
2025-09-15 19:32:28,905 - NEW difference columns
enc_paym_avg_0_1_this_year_diff
enc_paym_avg_1_2_all_diff
enc_paym_avg_0_years_diff

2025-09-15 19:32:29,554 - DataFrame shape after drop(): (5231241, 35)
2025-09-15 19:32:29,627

array([[0.62196784, 0.37803216],
       [0.42176404, 0.57823596],
       [0.50471337, 0.49528663],
       ...,
       [0.89598564, 0.10401436],
       [0.47170083, 0.52829917],
       [0.14379824, 0.85620176]])

In [13]:
# Вычислим целевую метрику
roc_auc_score(train_test_dict['y_test'], pred_proba[:,1])

0.7556375391892574

Метрика roc_auc_score получилась практически такой же как и в исследовательском ноутбуке (0.7572337893426191),
думаю разницу можно объяснить погрешностью вычислений, к тому же схемы сбора тренировочных датасетов немного отличается.
В исследовательской части мы набирали признаки  из исходного датасета в датасет с таргетом размером (2400000, 1) а в ноутбуке пайплайна сразу в исходный датасет  размером (20931476, N) и после удаляли дубликаты строк чтобы привести в соответствие с размером таргет датасета.


In [14]:
# Отключим логирование и предскажем класс 1
pred = main_pipe.predict(train_test_dict['X_test'])
pred

2025-09-15 19:33:00,757 - FUNCTION convert_all_to_numeric_preprocessing
2025-09-15 19:33:01,416 - FUNCTION cast_columns_by_map_preprocessing
2025-09-15 19:33:01,682 - FUNCTION drop_duplicates_preprocessing
2025-09-15 19:33:07,726 - No duplicates found, no cleanup operation required.
2025-09-15 19:33:07,727 - FUNCTION rn_max_feature_pipeline
2025-09-15 19:33:08,923 - FUNCTION enc_paym_transcoding_pipeline
2025-09-15 19:33:09,873 - FUNCTION enc_paym_norm_group_sum_diff_pipeline
2025-09-15 19:33:09,873 - NEW temporary features
2025-09-15 19:33:10,158 - enc_paym_avg_1_all
2025-09-15 19:33:10,695 - enc_paym_avg_2_all
2025-09-15 19:33:11,273 - enc_paym_avg_0_this_year
2025-09-15 19:33:11,635 - enc_paym_avg_1_this_year
2025-09-15 19:33:12,013 - enc_paym_avg_0_last_year
2025-09-15 19:33:12,553 - NEW difference columns
enc_paym_avg_0_1_this_year_diff
enc_paym_avg_1_2_all_diff
enc_paym_avg_0_years_diff

2025-09-15 19:33:13,198 - DataFrame shape after drop(): (5231241, 35)
2025-09-15 19:33:13,265

array([0, 1, 1, ..., 0, 1, 1])

In [15]:
# Проверим метод трансформера также без логирования
X_test_transformed = main_pipe.transform(train_test_dict['X_test'])
X_test_transformed

2025-09-15 19:33:42,752 - FUNCTION convert_all_to_numeric_preprocessing
2025-09-15 19:33:43,360 - FUNCTION cast_columns_by_map_preprocessing
2025-09-15 19:33:43,655 - FUNCTION drop_duplicates_preprocessing
2025-09-15 19:33:49,489 - No duplicates found, no cleanup operation required.
2025-09-15 19:33:49,490 - FUNCTION rn_max_feature_pipeline
2025-09-15 19:33:50,680 - FUNCTION enc_paym_transcoding_pipeline
2025-09-15 19:33:51,657 - FUNCTION enc_paym_norm_group_sum_diff_pipeline
2025-09-15 19:33:51,658 - NEW temporary features
2025-09-15 19:33:51,925 - enc_paym_avg_1_all
2025-09-15 19:33:52,469 - enc_paym_avg_2_all
2025-09-15 19:33:53,031 - enc_paym_avg_0_this_year
2025-09-15 19:33:53,402 - enc_paym_avg_1_this_year
2025-09-15 19:33:53,767 - enc_paym_avg_0_last_year
2025-09-15 19:33:54,300 - NEW difference columns
enc_paym_avg_0_1_this_year_diff
enc_paym_avg_1_2_all_diff
enc_paym_avg_0_years_diff

2025-09-15 19:33:54,949 - DataFrame shape after drop(): (5231241, 35)
2025-09-15 19:33:55,015

,rn_max,enc_paym_avg_0_1_this_year_diff,enc_paym_avg_1_2_all_diff,enc_paym_avg_0_years_diff,pre_util_mean_freq,pre_loans_credit_limit_mean_freq,pre_since_opened_mean_freq,pre_loans_credit_cost_rate_mean_freq,enc_loans_credit_type_mean_freq,pre_loans_next_pay_summ_mean_freq,...,pre_util_prop_6,pre_since_confirmed_prop_4,pre_since_confirmed_prop_7,pre_pterm_prop_6,pre_pterm_prop_3,enc_loans_account_holder_type_prop_4,pre_till_pclose_prop_10,pre_till_pclose_prop_7,is_zero_sum_prop_1,pre_since_opened_repeated_prop
0,15,10.266667,-7.000000,4.133333,0.390599,0.051324,0.052867,0.111763,0.422581,0.428804,...,0.066667,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.200000,1.000000,0.466667
1,9,9.333333,-11.000000,7.000000,0.330802,0.042885,0.049570,0.118809,0.356251,0.192153,...,0.222222,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.977778,0.111111
2,1,5.000000,-17.000000,5.000000,0.009697,0.050955,0.050015,0.062698,0.564650,0.052886,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.000000,0.000000
3,18,10.333333,-7.500000,5.055556,0.594809,0.057809,0.054065,0.220353,0.549710,0.506079,...,0.000000,0.222222,0.166667,0.000000,0.055556,0.0,0.000000,0.000000,0.911111,0.500000
4,5,6.600000,-15.600000,6.600000,0.433509,0.049831,0.046640,0.095862,0.298481,0.304307,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.960000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,18,6.666667,-12.388889,2.944444,0.528509,0.053334,0.050186,0.196005,0.460068,0.610150,...,0.000000,0.055556,0.388889,0.166667,0.055556,0.0,0.055556,0.000000,0.988889,0.500000
599996,17,9.705882,-7.411765,3.588235,0.511754,0.050765,0.050801,0.100580,0.438096,0.458600,...,0.058824,0.058824,0.058824,0.000000,0.000000,0.0,0.000000,0.058824,0.917647,0.294118
599997,12,10.250000,-7.583333,4.916667,0.545731,0.055853,0.047204,0.212760,0.341056,0.471056,...,0.083333,0.083333,0.166667,0.166667,0.000000,0.0,0.000000,0.083333,0.983333,0.333333
599998,10,9.900000,-9.700000,6.800000,0.431285,0.050851,0.046815,0.067475,0.483972,0.428646,...,0.100000,0.000000,0.000000,0.100000,0.000000,0.0,0.000000,0.000000,0.920000,0.400000


In [16]:
# Сохраним предсказания классов и их вероятностей в файлы

# Создаём имя файла предикта вероятностей
proba_file_name = make_file_path(
    output_type='proba',
    data_path=RAW_DATA_PATH,
    output_dir=TEST_PREDICT_PATH,
    ext=SAVE_FILE_EXTENSION
)

# Создаём имя файла предикта меток классов
predict_file_name = make_file_path(
    output_type='predict',
    data_path=RAW_DATA_PATH,
    output_dir=TEST_PREDICT_PATH,
    ext=SAVE_FILE_EXTENSION
)


# Получаем id set для сохранения с предиктом
# Используем drop_duplicates так как X_test это датасет до агрегаций в пайплайне
ids = train_test_dict['X_test']['id'].drop_duplicates().values

# Сохраненяем вероятности в .csv
save_predictions_with_id(
    output_type='proba',
    ids=ids,
    predictions=pred_proba,
    output_path=proba_file_name
)

# Сохраненяем метки классов в .csv
save_predictions_with_id(
    output_type='predict',
    ids=ids,
    predictions=pred,
    output_path=predict_file_name
)

2025-09-15 19:34:21,309 - Started save_predictions_with_id function
2025-09-15 19:34:23,433 - Started save_predictions_with_id function
